# Setup

In [ ]:
import analysis as al
import matplotlib.pyplot as plt
import numpy as np
import torch
import tqdm.auto as tqdm
%matplotlib widget

In [ ]:
torch.set_default_device('cpu')
torch.set_default_dtype(torch.float64)
train_device = 'cpu'

In [ ]:
def grab(x):
    return x.detach().cpu().numpy()

In [ ]:
def init_field(batch, Nc, Lx, Lt, dtype=torch.float64):
    """Uniform random field on CP(N-1): normalise real 2Nc-vector."""
    x = torch.randn(batch, 2*Nc, Lx, Lt, dtype=dtype)
    x = x / x.norm(dim=1, keepdim=True)
    return x

In [ ]:
def to_complex(x, axis=1):
    ind0, ind1 = slice(0, None, 2), slice(1, None, 2)
    if axis < 0:
        axis += len(x.shape)
    ind0 = (slice(None),)*axis + (ind0,)
    ind1 = (slice(None),)*axis + (ind1,)
    XR = x[ind0] # x[:,::2]
    XI = x[ind1] # x[:,1::2]
    z = XR + 1j*XI
    zbar = XR - 1j*XI
    return z, zbar
def to_real(z, axis=1):
    if axis < 0:
        axis += len(z.shape)
    return torch.stack([z.real, z.imag], axis=axis+1).flatten(axis,axis+1)
def _test_real_complex():
    x = torch.randn((3, 4, 6))
    assert torch.allclose(to_real(to_complex(x)[0]), x)
    z = to_complex(x, 1)[0]
    assert torch.allclose(to_real(z, 1), x)
    print(f'[PASSED test_real_complex]')
# _test_real_complex()
def action(z, zbar, *, beta):
    assert len(z.shape) == 4, 'z must have shape (batch, Nc, Lx, Lt)'
    assert z.shape == zbar.shape
    S = torch.zeros(z.shape[0])
    for mu in range(2):
        h1 = torch.sum(z * torch.roll(zbar, -1, dims=mu+2), axis=1)
        h2 = torch.sum(zbar * torch.roll(z, -1, dims=mu+2), axis=1)
        S = S + torch.sum(1.0 - h1*h2, axis=(1,2))
    return beta * S
def grad_action(x, *, beta):
    assert len(x.shape) == 4, 'x must have shape (batch, 2*Nc, Lx, Lt)'
    def _single_action(x):
        z, zbar = to_complex(x, axis=0)
        return action(z[None], zbar[None], beta=beta)[0].real
    gS = torch.func.vmap(torch.func.jacrev(_single_action))(x)
    assert gS.shape == x.shape
    inds = tuple(range(1, len(gS.shape)))
    assert torch.allclose((x*x).sum(1), torch.tensor(1.0).to(x))
    gS -= x * (gS * x).sum(1, keepdim=True)
    _dot = (gS * x).sum(1)
    assert torch.allclose(_dot, torch.tensor(0.0).to(x), atol=1e-5), f'{_dot.dtype=} {gS.dtype=} {x.dtype=} {_dot=}'
    return gS
def grad_action2(x, *, beta):
    assert len(x.shape) == 4, 'x must have shape (batch, 2*Nc, Lx, Lt)'
    gS = torch.zeros_like(x)
    for mu in range(2):
        for d in [-1, 1]:
            w = torch.roll(x, d, dims=mu+2)
            ow = to_real(1j*to_complex(w, axis=1)[0], axis=1)
            gS -= 2 * beta * (w*(w*x).sum(1, keepdim=True) + ow*(ow*x).sum(1, keepdim=True))
    gS -= x * (gS * x).sum(1, keepdim=True)
    _dot = (gS * x).sum(1)
    assert torch.allclose(_dot, torch.tensor(0.0).to(x), atol=1e-5), f'{_dot.dtype=} {gS.dtype=} {x.dtype=} {_dot=}'
    return gS
def _compare_grad_action():
    x = init_field(5, Nc=3, Lx=8, Lt=8)
    g1 = grad_action(x, beta=2.0)
    g2 = grad_action2(x, beta=2.0)
    assert torch.allclose(g1, g2), f'{g1=} {g2=}'
    print('[PASSED compare_grad_action]')
_compare_grad_action()

# HMC

In [ ]:
def leapfrog(x, p, *, n_steps, dtau, grad_action):
    """
    Leapfrog on the CP(N-1) constraint manifold (tangent-space momenta).

    x : (batch, 2Nc, Lx, Lt)  – field (unit-norm along dim=1)
    p : (batch, 2Nc, Lx, Lt)  – momentum (tangent: p·x = 0 per site)
    """
    def project(p, x):
        """Project p onto tangent space of the sphere at x."""
        return p - x * (p * x).sum(1, keepdim=True)

    def retract(x, v, eps):
        """Geodesic retraction: move along great circle, re-normalise."""
        norm_v = v.norm(dim=1, keepdim=True).clamp(min=1e-30)
        x_new = x * torch.cos(eps*norm_v) + (v / norm_v) * torch.sin(eps*norm_v)
        v_new = v * torch.cos(eps*norm_v) - (x * norm_v) * torch.sin(eps*norm_v)
        x_new = x_new / x_new.norm(dim=1, keepdim=True)   # safety renorm
        return x_new, v_new

    # leapfrog steps
    gS = grad_action(x)
    p = p - 0.5 * dtau * gS
    for _ in range(n_steps - 1):
        x, p = retract(x, p, dtau)
        gS = grad_action(x)
        p = p - dtau * gS
    x, p = retract(x, p, dtau)
    gS = grad_action(x)
    p = p - 0.5 * dtau * gS

    return x, p

def hmc_traj(x, *, theory, n_steps, dtau, ar=True):
    batch = x.shape[0]
    action, grad_action = theory

    # tangent-space momenta
    p = torch.randn_like(x)
    p = p - x * (p * x).sum(1, keepdim=True)

    K_old = 0.5 * (p * p).sum(dim=(1, 2, 3))
    z, zbar = to_complex(x, axis=1)
    S_old = action(z, zbar).real

    x_prop, p_prop = leapfrog(x, p, n_steps=n_steps, dtau=dtau, grad_action=grad_action)

    if not ar:
        return x_prop

    K_new = 0.5 * (p_prop * p_prop).sum(dim=(1, 2, 3))
    z_p, zbar_p = to_complex(x_prop, axis=1)
    S_new = action(z_p, zbar_p).real

    dH = (S_new + K_new) - (S_old + K_old)
    accepted = torch.rand(batch) < torch.exp(-dH).clamp(max=1.0)
    x_out = torch.where(accepted[:, None, None, None], x_prop, x)
    return x_out, accepted

def run_hmc(x0, *, theory, n_steps, dtau, n_traj, n_therm=0):
    """Run batched HMC"""
    x = x0.clone()
    acc_tot = 0

    for _ in tqdm.tqdm(range(n_therm)):
        x, acc = hmc_traj(x, theory=theory, n_steps=n_steps, dtau=dtau)
c
    ens = []
    for _ in tqdm.tqdm(range(n_traj)):
        x, acc = hmc_traj(x, theory=theory, n_steps=n_steps, dtau=dtau)
        ens.append(x.clone())
        acc_tot += acc.float().mean().item()

    ens = torch.stack(ens, dim=0)
    acc_rate  = acc_tot / n_traj
    return ens, acc_rate

In [ ]:
def _check_hmc():
    torch.manual_seed(42)
    Nc, Lx, Lt = 3, 32, 32
    batch = 1
    beta = 4.0
    theory = (
        lambda z, zbar: action(z, zbar, beta=beta),
        lambda x: grad_action2(x, beta=beta),
    )

    x0 = init_field(batch, Nc, Lx, Lt)

    ens, acc = run_hmc(
        x0,
        theory = theory,
        n_steps = 20,
        dtau = 0.05,
        n_traj = 20,
        n_therm = 100,
    )

    print(f'Ensemble shape : {ens.shape}')   # (20, 4, 6, 8, 8)
    print(f'Acceptance rate: {acc:.3f}')

    # sanity: unit norm preserved
    norms = ens.norm(dim=2)   # norm over 2Nc channel
    print(f'Max |‖z‖−1|   : {(norms - 1).abs().max():.2e}')

    z, zbar = to_complex(ens[-1,0], axis=0)
    fig, ax = plt.subplots(1,1)
    ax.imshow(np.angle(z[1]/z[2]), vmin=-np.pi, vmax=np.pi, cmap='twilight')
    plt.show()
_check_hmc()

# HAD

In [ ]:
def run_had(x0, *, theory, n_steps, dtau, n_traj, n_therm=0):
    """Run batched HMC"""
    x = x0.clone()
    b = torch.zeros_like(x)

    for _ in tqdm.tqdm(range(n_therm)):
        x, acc = hmc_traj(x, theory=theory, n_steps=n_steps, dtau=dtau, ar=True)

    ens = []
    bs = []
    for _ in tqdm.tqdm(range(n_traj)):
        x, b = torch.func.jvp(
            lambda x: hmc_traj(x, theory=theory, n_steps=n_steps, dtau=dtau, ar=False),
            (x,), (b,),
        )
        ens.append(x.clone())
        bs.append(b.clone())

    ens = torch.stack(ens, dim=0)
    bs = torch.stack(bs, dim=0)
    return ens, bs

In [ ]:
def _check_had():
    torch.manual_seed(42)
    Nc, Lx, Lt = 3, 16, 16
    batch = 4
    beta = 2.0
    theory = (
        lambda z, zbar: action(z, zbar, beta=beta),
        lambda x: grad_action2(x, beta=beta),
    )

    x0 = init_field(batch, Nc, Lx, Lt)

    ens, bs = run_had(
        x0,
        theory = theory,
        n_steps = 250,
        dtau = 0.05,
        n_traj = 20,
        n_therm = 50,
    )

    ens = ens.flatten(0,1)
    bs = bs.flatten(0,1)

    print(f'Ensemble shape : {ens.shape}')   # (20, 4, 6, 8, 8)

    z, zbar = to_complex(grab(ens), axis=1)
    b, bbar = to_complex(grab(bs), axis=1)
    Cij = z[:,0] * zbar[:,1] * zbar[:,0] * 

    bs = np.mean(grab(bs), axis=(0,1))
    print(f'{bs.shape=}')

    z, zbar = to_complex(bs, axis=0)
    fig, ax = plt.subplots(1,1)
    ax.imshow(np.angle(z[1]/z[2]), vmin=-np.pi, vmax=np.pi, cmap='twilight')
    plt.show()
_check_had()